In [2]:
import os, sys

# Repository information
REPO_NAME = "RecSys-Challenge-2025"
REPO_URL  = f"github.com/Lv1g1/{REPO_NAME}.git"

# Detect environment
IS_COLAB = 'content' in os.getcwd()
IS_KAGGLE = 'kaggle' in os.getcwd()
IS_LOCAL = not (IS_COLAB or IS_KAGGLE)

WORKING_DIR = os.getcwd()

if IS_COLAB:
    WORKING_DIR = "/content"

    # Mount Google Drive
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)

    # Get GitHub token via input
    def get_token():
        from getpass import getpass
        return getpass("GitHub Token: ")

elif IS_KAGGLE:
    WORKING_DIR = "/kaggle/working"

    # Get GitHub token from Kaggle secrets
    def get_token():
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("Token")

# If local environment assume inside the repo
LOCAL_REPO_PATH = "/home/luigi/RecSys" if IS_LOCAL else os.path.join(WORKING_DIR, REPO_NAME)

# Clone the repository if it doesn't exist
if not os.path.exists(LOCAL_REPO_PATH):
    os.chdir(WORKING_DIR)
    token = get_token()

    !git clone https://{token}@{REPO_URL}
else:
    print("Repo already exists — pulling latest changes")
    os.chdir(LOCAL_REPO_PATH)
    !git pull
    os.chdir(WORKING_DIR)

# Add to Python PATH
if LOCAL_REPO_PATH not in sys.path:
    sys.path.append(LOCAL_REPO_PATH)

Repo already exists — pulling latest changes
Already up to date.


In [3]:
import importlib
import scipy.sparse as sps
import pandas as pd
import numpy as np

from Challenge import paths
importlib.reload(paths)

Running on local — storage at: /home/luigi/RecSys
Running on local — storage at: /home/luigi/RecSys


<module 'Challenge.paths' from '/home/luigi/RecSys/Challenge/paths.py'>

In [4]:
# Load datasets
folds = paths.load_cv_folds(k=5)

In [5]:
# Load from disk cached scores
import pickle

filepath = os.path.join(paths.MODEL_DIR, "hybrid", "cached_scores.pkl")
with open(filepath, 'rb') as f:
    cached_scores = pickle.load(f)

In [6]:
for idx in range(5):
    for user_id in range(len(cached_scores[idx])):
        cached_scores[idx][user_id]['recommended_items'] = np.array(cached_scores[idx][user_id]['recommended_items'])

In [7]:
def get_recommendation(user_id, w, idx, cutoff=20):
    score = cached_scores[idx][user_id]['scores_matrix'] @ w
    items = cached_scores[idx][user_id]['recommended_items']
    ranking_indices = np.argsort(-score)[:cutoff]
    return items[ranking_indices]

In [8]:
models = ['SLIM', 'EASE_R', 'TopPop', 'UserKNNcosine', 'UserKNNpearson', 'UserKNNjaccard', 'UserKNNtversky', 'ItemKNNcosine', 'ItemKNNpearson', 'ItemKNNjaccard', 'ItemKNNtversky']

In [9]:
import optuna

def objective_function(optuna_trial: optuna.trial.Trial) -> float:
    weights = [
        optuna_trial.suggest_float(f"weight_{i}", 0.0, 0.3)
        for i in range(10)
    ]
    weights.insert(0, 1.0) # Max to SLIM

    # Normalization isn't strictly necessary here
    # Doesn't change the relative importance of models
    # and so the final ranking
    # Print normalized because are more interpretable weights
    # w_to_model = {models[i]: w / sum(weights) for i,w in enumerate(weights)}
    # # sort by weights
    # w_to_model = dict(sorted(w_to_model.items(), key=lambda item: item[1], reverse=True))
    # # pretty print
    # print("Current weights:")
    # for key, w in w_to_model.items():
    #     print(f"  {key}: {w:.4f}")
    # print()

    # Convert weights to numpy array
    weights = np.array(weights)
    
    validation_scores = []
    for idx, (_, URM_validation) in enumerate(folds):        
        # Evaluate weight configuration on validation set
        cumulative_recall = 0.0
        num_eval = 0
        
        for user_id in range(URM_validation.shape[0]):
            relevant_items = URM_validation.indices[URM_validation.indptr[user_id]:URM_validation.indptr[user_id+1]]
            
            if len(relevant_items)>0:
                num_eval+=1
                
                recommended_items = get_recommendation(user_id, weights, idx, cutoff=20)
                
                is_relevant = np.isin(recommended_items, relevant_items, assume_unique=True)
                recall_score = np.sum(is_relevant, dtype=np.float32) / relevant_items.shape[0]

                cumulative_recall += recall_score

        recall = cumulative_recall / num_eval

        # Store fold result
        validation_scores.append(recall)

    avg_score = np.mean(validation_scores)
    # print(f"Average validation score: {avg_score:.4f}")

    return avg_score

In [10]:
STUDY_NAME = "hybrid_score_blending_optimization_3"

In [11]:
optuna_study = optuna.create_study(
    study_name=STUDY_NAME,
    storage=paths.OPTUNA_STORAGE,
    direction="maximize",
    load_if_exists=True
)

[I 2025-11-10 02:21:00,733] Using an existing study with name 'hybrid_score_blending_optimization_3' instead of creating a new one.


In [12]:
optuna_study.optimize(
    objective_function,
    n_trials=2000,
    show_progress_bar=True
)

  0%|          | 0/2000 [00:00<?, ?it/s]

[I 2025-11-10 02:22:17,199] Trial 116 finished with value: 0.09493328630924225 and parameters: {'weight_0': 0.2242497796242661, 'weight_1': 0.021536019340799503, 'weight_2': 0.08942707084079003, 'weight_3': 0.024288373518927302, 'weight_4': 0.12440187321312124, 'weight_5': 0.27315971751454254, 'weight_6': 0.012407676153801506, 'weight_7': 0.17282900661310677, 'weight_8': 0.2171849188171389, 'weight_9': 0.051421380886445864}. Best is trial 98 with value: 0.15145450830459595.
[I 2025-11-10 02:22:20,534] Trial 117 finished with value: 0.10471727699041367 and parameters: {'weight_0': 0.24696160970107678, 'weight_1': 0.008493483768567114, 'weight_2': 0.10900638984666819, 'weight_3': 0.009056305757372296, 'weight_4': 0.13726863495164346, 'weight_5': 0.27608791048980064, 'weight_6': 0.08232471496104189, 'weight_7': 0.15110225470425515, 'weight_8': 0.22655827363566303, 'weight_9': 0.12757121688072934}. Best is trial 98 with value: 0.15145450830459595.
[I 2025-11-10 02:22:23,882] Trial 118 fini

KeyboardInterrupt: 

In [ ]:
optuna.visualization.plot_optimization_history(optuna_study)

In [ ]:
optuna.visualization.plot_param_importances(optuna_study)

In [ ]:
optuna.visualization.plot_parallel_coordinate(optuna_study)